In [5]:
import pandas as pd
import os
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from langid import classify
from tqdm import tqdm

tqdm.pandas()

In [6]:
# 1. Initialization & Setup

try:
    nltk.data.find("vader_lexicon")
except LookupError:
    nltk.download("vader_lexicon")

print("Initializing VADER and loading custom Airbnb lexicon...")
SIA = SentimentIntensityAnalyzer()

# Custom lexicon
custom_lexicon = {
    "cancel": -20, "canceled": -20, "canceling": -20,
    "backpain": -5, "bad": -10, "spiderwebs": -50,
    "odor": -30, "freaked": -30, "musty": -50,
    "toxic": -5, "sticky": -15, "ugly": -15,
    "bedbugs": -60, "bugs": -20, "rude": -30,
    "aggressive": -30, "scary": -15, "cozy": 10,
    "great": 20, "cosy": 10, "smoothly": 30,
    "worst": -10, "convenient": 40, "worse": -10,
    "exciting": 60, "notch": 30, "superhost": 30,
    "disappointing": -10, "horrible": -30, "dirty": -15,
    "dirt": -15, "stain": -20, "filthy": -30,
    "unreliable": -15, "meh": -5, "spacious": 10,
    "lovely": 20, "infested": -15, "broke": -10,
    "broken": -15, "awake": -20, "difficult": -20,
    "1010": 10,
}
SIA.lexicon.update(custom_lexicon)

Initializing VADER and loading custom Airbnb lexicon...


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/gonghuan/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [7]:
# 2. Functions

def clean_text_optimized(text):
    """Strip HTML tags, HTML entities, and normalise whitespace."""
    text = str(text)

    # Strip HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Strip HTML entities (&amp; &#39; &quot; etc.)
    text = re.sub(r"&[a-zA-Z]+;", " ", text)
    text = re.sub(r"&#\d+;",     " ", text)
    
    # Normalise whitespace
    text = " ".join(text.split())
    return text

def is_english(text):
    """Return True if langid classifies text as English."""
    try:
        lang, _ = classify(text)
        return lang == "en"
    except Exception:
        return False

def get_vader_score(sentence):
    return SIA.polarity_scores(sentence)["compound"]

In [8]:
# 3. Main Execution Pipeline

if __name__ == "__main__":
    print("\n--- Starting Large-Scale Analysis Pipeline ---")

    folder_path     = "/Users/gonghuan/Downloads/MergedEdinburgh"
    target_filename = "2024-10-18_reviews.csv.gz"
    full_file_path  = os.path.join(folder_path, target_filename)

    if not os.path.exists(full_file_path):
        print(f"[Error] File not found: {full_file_path}")
    else:
        # Load data
        df = pd.read_csv(full_file_path, usecols=["listing_id", "id", "comments"])
        print(f"Loaded initial data: {len(df)} rows.\n")

        # Step 1: Text Cleaning & Filtering
        
        print("--- Step 1/4: Cleaning text & filtering short comments ---")

        # 1.1 Drop NaN comments
        initial_len = len(df)
        df = df.dropna(subset=["comments"])
        print(f"  Dropped {initial_len - len(df)} rows with missing comments. "
              f"Remaining: {len(df)}")

        # 1.2 Text cleaning (HTML tags + entities + whitespace)
        print("  Applying text cleaning (HTML tags + entities + whitespace)...")
        df["clean_text"] = df["comments"].progress_apply(clean_text_optimized)

        # Show examples
        print("\n  Sample text cleaning (before vs after):")
        changed_samples = df[df["comments"] != df["clean_text"]].head(3)
        for _, row in changed_samples.iterrows():
            print(f"    Before : {repr(row['comments'][:100])}...")
            print(f"    After  : {repr(row['clean_text'][:100])}...\n")

        # 1.3 Drop short comments
        pre_short_len = len(df)
        df = df[df["clean_text"].str.len() > 3]
        print(f"  Dropped {pre_short_len - len(df)} rows with length <=3 chars. "
              f"Remaining: {len(df)}\n")

        
        # Step 2: English-only filter
        
        print("--- Step 2/4: English-only filter ---")
        pre_lang_len = len(df)
        df["is_en"] = df["clean_text"].progress_apply(is_english)

        non_en_count = (~df["is_en"]).sum()
        print(f"\n  Non-English reviews detected: {non_en_count} "
              f"({non_en_count/len(df)*100:.1f}%)")

        df = df[df["is_en"]].drop(columns=["is_en"]).copy()
        print(f"  After English filter: {len(df)} rows "
              f"(dropped {pre_lang_len - len(df)})\n")

        
        # Step 3: Sentiment Analysis
        
        print("--- Step 3/4: Running Custom VADER Sentiment Analysis ---")
        df["sentiment_score"] = df["clean_text"].progress_apply(get_vader_score)

        print("\n  Sample sentiment scores:")
        for _, row in df.head(3).iterrows():
            print(f"    [Text]  : {repr(row['clean_text'][:100])}...")
            print(f"    [Score] : {row['sentiment_score']}\n")

        print("  Score distribution:")
        print(df["sentiment_score"].describe().to_string())

        
        # Step 4: Aggregate per listing
        
        print("\n--- Step 4/4: Aggregating results per listing ---")
        initial_listings = df["listing_id"].nunique()
        print(f"  Total unique listings before aggregation: {initial_listings}")

        listing_sentiment = df.groupby("listing_id").agg({
            "sentiment_score": "mean",
            "id":              "count"
        }).rename(columns={"sentiment_score": "average_sentiment",
                           "id":              "review_count"})

        pre_filter_listings = len(listing_sentiment)
        listing_sentiment = listing_sentiment[listing_sentiment["review_count"] >= 5]
        listing_sentiment = listing_sentiment.sort_values("average_sentiment", ascending=False)

        print(f"  Dropped {pre_filter_listings - len(listing_sentiment)} "
              f"listings with <5 reviews.")
        print(f"\n  Final valid listings: {len(listing_sentiment)}")

        
        # Step 5: Save
        
        output_dir      = "/Users/gonghuan/Downloads"
        output_filename = "airbnb_sentiment_results_new_huan.csv"
        output_path     = os.path.join(output_dir, output_filename)

        listing_sentiment.to_csv(output_path)
        print(f"\n  Saved: {output_path}")

        # Sanity check
        print("\n--- Sanity check ---")
        print(f"  Listing count: {len(listing_sentiment)}")
        print(f"  average_sentiment stats:")
        print(f"    min    : {listing_sentiment['average_sentiment'].min():.4f}")
        print(f"    max    : {listing_sentiment['average_sentiment'].max():.4f}")
        print(f"    mean   : {listing_sentiment['average_sentiment'].mean():.4f}")
        print(f"    median : {listing_sentiment['average_sentiment'].median():.4f}")
        print(f"    std    : {listing_sentiment['average_sentiment'].std():.4f}")



--- Starting Large-Scale Analysis Pipeline ---
Loaded initial data: 551523 rows.

--- Step 1/4: Cleaning text & filtering short comments ---
  Dropped 44 rows with missing comments. Remaining: 551479
  Applying text cleaning (HTML tags + entities + whitespace)...


100%|██████████| 551479/551479 [00:02<00:00, 258570.82it/s]



  Sample text cleaning (before vs after):
    Before : 'My wife and I stayed at this beautiful apartment and our stay was spectacular.  The neighborhood is '...
    After  : 'My wife and I stayed at this beautiful apartment and our stay was spectacular. The neighborhood is v'...

    Before : "Charlotte couldn't have been a more thoughtful or accomodating host.\r<br/>The flat had literally eve"...
    After  : "Charlotte couldn't have been a more thoughtful or accomodating host. The flat had literally everythi"...

    Before : "This flat was incredible. As other guests have mentioned, Charlotte's attention to detail is unrival"...
    After  : "This flat was incredible. As other guests have mentioned, Charlotte's attention to detail is unrival"...

  Dropped 1659 rows with length <=3 chars. Remaining: 549820

--- Step 2/4: English-only filter ---


100%|██████████| 549820/549820 [15:17<00:00, 599.44it/s]  



  Non-English reviews detected: 73523 (13.4%)
  After English filter: 476297 rows (dropped 73523)

--- Step 3/4: Running Custom VADER Sentiment Analysis ---


100%|██████████| 476297/476297 [01:43<00:00, 4603.66it/s]



  Sample sentiment scores:
    [Text]  : 'My wife and I stayed at this beautiful apartment and our stay was spectacular. The neighborhood is v'...
    [Score] : 0.9816

    [Text]  : "Charlotte couldn't have been a more thoughtful or accomodating host. The flat had literally everythi"...
    [Score] : 0.8583

    [Text]  : "I went to Edinburgh for the second time on April 2011 and, also thanks to Charlotte's flat, the stay"...
    [Score] : 0.9959

  Score distribution:
count    476297.000000
mean          0.895362
std           0.261610
min          -0.999900
25%           0.925100
50%           0.986000
75%           0.994800
max           0.999900

--- Step 4/4: Aggregating results per listing ---
  Total unique listings before aggregation: 5174
  Dropped 602 listings with <5 reviews.

  Final valid listings: 4572

  Saved: /Users/gonghuan/Downloads/airbnb_sentiment_results_new_huan.csv

--- Sanity check ---
  Listing count: 4572
  average_sentiment stats:
    min    : -0.5028
    